# California Housing KNN Regression GridSearchCV
**info:** Use the California Housing dataset. Keep the KNN Regression
Pipeline structure with StandardScaler, the hyperparameter grid, GridSearchCV configuration with KFold, and
regression evaluation workflow unchanged.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
def evaluate_regression(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    print(f"\n========= {name} Evaluation =========")
    print(f'R² Score: {r2:.2f}')
    print(f'Mean Absolute Error: {mae:.2f}')
    print(f'Mean Squared Error: {mse:.2f}')
    print(f'Root Mean Squared Error: {rmse:.2f}')

## 2. Load California Housing Dataset

In [3]:
california = fetch_california_housing()

In [4]:
X = california.data
y = california.target

feature_names = california.feature_names

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f"Features: {feature_names}")

X shape: (20640, 8)
y shape: (20640,)
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


## 3. Split Data

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)
print(f'Train shape: {X_train.shape[0]}')
print(f'Test shape: {X_test.shape[0]}')

Train shape: 16512
Test shape: 4128


## 4. Create KNN Regression Pipeline

In [6]:
KNN_Pipeline = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor())
    ]
)

## 5. Hyperparameter Tunin

In [7]:
params = {
    'knn__n_neighbors': range(1, 30),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ["euclidean", "manhattan", "chebyshev"]
}

kf = KFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=KNN_Pipeline,
    param_grid=params,
    scoring='neg_mean_squared_error',
    cv=kf,
    verbose=1
)
grid.fit(X_train, y_train)

Fitting 10 folds for each of 174 candidates, totalling 1740 fits


GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn', KNeighborsRegressor())]),
             param_grid={'knn__metric': ['euclidean', 'manhattan', 'chebyshev'],
                         'knn__n_neighbors': range(1, 30),
                         'knn__weights': ['uniform', 'distance']},
             scoring='neg_mean_squared_error', verbose=1)

## 6. Best Parameters

In [8]:
print("Best Parameters:", grid.best_params_)

print("\nBest CV RMSE:")
print(-grid.best_score_)

Best Parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 11, 'knn__weights': 'distance'}

Best CV RMSE:
0.3566878574995552


## 7. Test Evaluation

In [9]:
best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test)

evaluate_regression('Best KNN', y_test, y_pred)


========= Best KNN Evaluation =========
R² Score: 0.72
Mean Absolute Error: 0.41
Mean Squared Error: 0.37
Root Mean Squared Error: 0.61


**Tip**

GridSearchCV evaluates different KNN Regression configurations using KFold cross-validation and selects the
best combination based on the chosen regression scoring metric. The best_estimator_ contains the complete
Pipeline with StandardScaler and KNeighborsRegressor